# CS781 Phase 5 — Legal Statute Identification (multi-label) from Phase 1 model 4 (masked-diffusion LM, bidirectional encoder; PyTorch, Kaggle GPU)
Phase 1 JAX weights → PyTorch port (adaLN at a fixed low noise level) → chunked hierarchical multi-label classifier (BCE + pos-weights) → LoRMA / Adapters / AdaLoRA / full fine-tuning → distillation into a half-depth student → global vs per-label threshold selection → one `submission` zip per experiment.

## 1. Setup and configuration

In [1]:
import os, sys, time, json, math, random, pickle, zipfile, gc, copy, glob, subprocess, importlib.util
T0 = time.time()
if importlib.util.find_spec("regex") is None: subprocess.run([sys.executable, "-m", "pip", "install", "-q", "regex"], check=False)
import numpy as np, regex as re2, torch, torch.nn as nn, torch.nn.functional as F
import multiprocessing as mp
from huggingface_hub import hf_hub_download, HfApi, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF"); login(token=HF_TOKEN)
except Exception as e:
    HF_TOKEN = None; print("HF login skipped:", e)

SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
HF_REPO = "Exploration-Lab/CS-781-Capstone"; WORK = "/kaggle/working/cs781_phase5_m4"; os.makedirs(WORK, exist_ok=True)
PIN = None
FOUND = [(p, pickle.load(open(p, "rb"))) for p in glob.glob("/kaggle/input/**/final_params.pkl", recursive=True)]
print("Phase 1 pickles found:", [(p, x["variant"]) for p, x in FOUND])
P1, d = next(((p, x) for p, x in FOUND if PIN in (None, p) and "adaln" in x["cfg"]), (None, None)); del FOUND
assert d is not None, "no ours_mdlm_* / mdlm_plain final_params.pkl attached: add the diffusion run's output as a Kaggle dataset (or set PIN to its path)"
print("using", P1, "->", d["variant"])
TOKP = os.path.join(os.path.dirname(P1), "bpe_legal.json")
if not os.path.exists(TOKP): TOKP = glob.glob("/kaggle/input/**/bpe_legal.json", recursive=True)[0]
TOTAL_SEC, POST_SEC = 5.6 * 3600, 8 * 60
CT, K, BS = 1024, 4, 4
NG = torch.cuda.device_count()
EXPS = [("lorma", 1.0), ("adapter", 0.9), ("adalora", 1.0), ("full", 1.1), ("kd", 1.2)]
LR = dict(lorma=5e-4, bitfit=1e-3, adapter=5e-4, adalora=5e-4, full=8e-5, kd=1e-4)
print("gpus:", NG, "|", P1, "|", TOKP)

Phase 1 pickles found: [('/kaggle/input/datasets/nirbhaykachhatiya/phase2m4/final_params.pkl', 'ours_mdlm_adaln_hybrid')]
using /kaggle/input/datasets/nirbhaykachhatiya/phase2m4/final_params.pkl -> ours_mdlm_adaln_hybrid
gpus: 2 | /kaggle/input/datasets/nirbhaykachhatiya/phase2m4/final_params.pkl | /kaggle/input/datasets/nirbhaykachhatiya/phase2m4/bpe_legal.json


## 2. Data: Phase 4 CJPE files, Phase 1 byte-level BPE, tokenised once

In [2]:
PAT = re2.compile(r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+")
def b2u():
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = b2u()

class BPE:
    def __init__(s, path):
        d = json.load(open(path, encoding="utf-8")); s.rank = {tuple(m): i for i, m in enumerate(d["merges"])}; s.stoi = {t: i for i, t in enumerate(d["itos"])}; s.c = {}
    def word(s, w):
        r = s.c.get(w)
        if r is None:
            x = [B2U[b] for b in w.encode("utf-8")]
            while len(x) > 1:
                best = min(zip(x, x[1:]), key=lambda q: s.rank.get(q, 1 << 60))
                if best not in s.rank: break
                a, b = best; y, i = [], 0
                while i < len(x):
                    if i < len(x) - 1 and x[i] == a and x[i + 1] == b: y.append(a + b); i += 2
                    else: y.append(x[i]); i += 1
                x = y
            r = s.c[w] = [s.stoi[t] for t in x]
        return r
    def encode(s, t): return [i for w in PAT.findall(t) for i in s.word(w)]

TK = BPE(TOKP)
def enc(ds): return [np.asarray(TK.encode(" ".join(d["text"].split())), np.uint16) for d in ds]

files = HfApi().list_repo_files(HF_REPO, repo_type="dataset", token=HF_TOKEN)
def load(key):
    fn = next(f for f in files if f.startswith("Phase_5_LSI/") and f.endswith(key))
    d = json.load(open(hf_hub_download(HF_REPO, fn, repo_type="dataset", token=HF_TOKEN, local_dir="/tmp/hf"), encoding="utf-8"))
    return [{"id": k, **(v if isinstance(v, dict) else {"text": v})} for k, v in d.items()] if isinstance(d, dict) else d

TOKC = f"{WORK}/tokens.pkl"
if os.path.exists(TOKC): TR, DV, TE, Y, DY, TE_ID, LAB = pickle.load(open(TOKC, "rb"))
else:
    tr, dv, te = load("train.json"), load("dev.json"), load("test.json")
    LK = next(k for k in ("labels", "label", "statutes", "y") if k in tr[0])
    LAB = sorted({int(l) for d in tr + dv for l in d[LK]}); IX = {l: i for i, l in enumerate(LAB)}
    def mh(ds):
        m = np.zeros((len(ds), len(LAB)), np.float32)
        for i, d in enumerate(ds):
            for l in d[LK]: m[i, IX[int(l)]] = 1
        return m
    Y, DY, TE_ID = mh(tr), mh(dv), [d["id"] for d in te]
    t0 = time.time()
    with mp.get_context("fork").Pool(os.cpu_count()) as pool:
        TR, DV, TE = [[a for p in pool.imap(enc, [ds[i:i + 100] for i in range(0, len(ds), 100)]) for a in p] for ds in (tr, dv, te)]
    pickle.dump((TR, DV, TE, Y, DY, TE_ID, LAB), open(TOKC, "wb")); del tr, dv, te; gc.collect(); print(f"tokenised in {time.time()-t0:.0f}s")
NL, NTR = len(LAB), len(TR); DYB = DY > 0.5; CNT = Y.sum(0)
PRI = np.clip(CNT / NTR, 1e-4, 0.5); PW = np.clip(np.sqrt((NTR - CNT) / np.maximum(CNT, 1)), 1, 30).astype(np.float32)
print(f"train/dev/test: {NTR}/{len(DV)}/{len(TE)} | statutes {NL} | labels/doc {Y.sum(1).mean():.2f} (max {int(Y.sum(1).max())}) | <10 train docs: {int((CNT < 10).sum())}, none: {int((CNT == 0).sum())} | median tokens {int(np.median([len(t) for t in TR]))}")
print("label freq quantiles (train docs):", np.quantile(CNT, [0, .1, .25, .5, .75, .9, 1]).astype(int).tolist())
SUB = np.random.RandomState(0).choice(len(DV), min(400, len(DV)), replace=False)
SELMODE = "even"

def sel(nc, rnd):
    if not rnd and SELMODE != "even": return [0] if SELMODE == "head" else [nc - 1]
    if nc <= K: return list(range(nc))
    mid = random.sample(range(nc - 1), K - 1) if rnd else np.linspace(0, nc - 2, K - 1).round().astype(int).tolist()
    return sorted(mid) + [nc - 1]

def mk(toks, idx, rnd):
    ids = np.zeros((len(idx), K, CT), np.int64); ln = np.zeros((len(idx), K), np.int64)
    for b, i in enumerate(idx):
        t = toks[i]; n = len(t)
        for k, c in enumerate(sel(max(1, -(-n // CT)), rnd)):
            x = t[min(c * CT, max(n - CT, 0)):][:CT]; ids[b, k, :len(x)] = x; ln[b, k] = len(x)
    ln = torch.from_numpy(ln).cuda()
    return torch.from_numpy(ids).cuda(), ln, ln > 0

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

tokenised in 51s
train/dev/test: 42750/10181/13019 | statutes 100 | labels/doc 3.70 (max 31) | <10 train docs: 0, none: 0 | median tokens 668
label freq quantiles (train docs): [272, 340, 472, 679, 1751, 3819, 11730]


## 3. Model: Phase 1 decoder ported to PyTorch, chunk encoder + attention pooling head, adaptation modules

In [3]:
P, C = d["params"], d["cfg"]; lp = P["layers"]; L, D, HHD = lp["wq"].shape; HD = lp["qn"].shape[1]; H = HHD // HD; KV = lp["wk"].shape[2] // HD; R = H // KV; V = P["emb"].shape[0]
CD = P["wt1"].shape[0] if C["adaln"] else 128; WIN, TC = 256, 0.05; LOC = [bool(C["hybrid"]) and i % 3 != 2 for i in range(L)]; LINS = ("wq", "wk", "wv", "wo", "w1", "w3", "w2")
INV = torch.tensor(1.0 / (10000.0 ** (np.arange(0, HD, 2) / HD)), dtype=torch.float32)
print(d["variant"], f"L={L} D={D} H={H} KV={KV} HD={HD} rows={V} adaln={C['adaln']} hybrid={C['hybrid']}", "|", torch.cuda.get_device_name(0))

def f32(w): return torch.from_numpy(np.asarray(w, np.float32).copy())
def lin(w):
    m = nn.Linear(w.shape[0], w.shape[1], bias=False); m.weight.data = torch.from_numpy(np.ascontiguousarray(w.T, dtype=np.float32)); return m

class RMS(nn.Module):
    def __init__(s, w): super().__init__(); s.w = nn.Parameter(f32(w))
    def forward(s, x): x = x.float(); return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + 1e-6) * s.w

def rope(x, cos, sin):
    a, b = x[..., :HD // 2].float(), x[..., HD // 2:].float()
    return torch.cat([a * cos - b * sin, b * cos + a * sin], -1)

class Blk(nn.Module):
    def __init__(s, i):
        super().__init__(); s.loc = LOC[i]
        for n in LINS: setattr(s, n, lin(lp[n][i]))
        s.n1, s.n2, s.qn, s.kn = RMS(lp["n1"][i]), RMS(lp["n2"][i]), RMS(lp["qn"][i]), RMS(lp["kn"][i]); s.wada = nn.Parameter(f32(lp["wada"][i])) if C["adaln"] else None
    def forward(s, x, cos, sin, km, lm, cond):
        B, T, _ = x.shape
        if cond is None: sh1 = sc1 = sh2 = sc2 = 0.0; g1 = g2 = 1.0
        else:
            with torch.autocast("cuda", enabled=False): z = (F.silu(cond) @ s.wada).chunk(6, -1)
            sh1, sc1, g1, sh2, sc2, g2 = [u[:, None] for u in z]; g1, g2 = 1 + g1, 1 + g2
        h = s.n1(x) * (1 + sc1) + sh1
        q, k, v = s.wq(h).view(B, T, H, HD), s.wk(h).view(B, T, KV, HD), s.wv(h).view(B, T, KV, HD)
        q, k = rope(s.qn(q), cos, sin).to(v.dtype), rope(s.kn(k), cos, sin).to(v.dtype)
        o = F.scaled_dot_product_attention(q.transpose(1, 2), k.repeat_interleave(R, 2).transpose(1, 2), v.repeat_interleave(R, 2).transpose(1, 2), attn_mask=lm if s.loc else km)
        x = x + g1 * s.wo(o.transpose(1, 2).reshape(B, T, H * HD)); h = s.n2(x) * (1 + sc2) + sh2
        return x + g2 * s.w2(F.silu(s.w1(h)) * s.w3(h))

class Enc(nn.Module):
    def __init__(s, keep):
        super().__init__(); s.emb = nn.Embedding(V, D); s.emb.weight.data = f32(P["emb"]); s.blocks = nn.ModuleList([Blk(i) for i in keep]); s.nf = RMS(P["nf"])
        t = torch.arange(CT)[:, None] * INV[None]; s.register_buffer("cos", t.cos()[None, :, None], persistent=False); s.register_buffer("sin", t.sin()[None, :, None], persistent=False)
        if C["adaln"]:
            for n in ("wt1", "bt1", "wt2", "bt2", "wadaf"): setattr(s, n, nn.Parameter(f32(P[n])))
            a = TC * 1000.0 * torch.exp(-math.log(10000.0) * torch.arange(CD // 2) / (CD // 2)); s.register_buffer("temb", torch.cat([a.cos(), a.sin()])[None], persistent=False)
    def forward(s, ids, ln):
        N, S = ids.shape; ar = torch.arange(S, device=ids.device)[None]; tm = ar < ln[:, None]
        km = (tm | (ar == 0))[:, None, None]; lm = km & ((ar.t() - ar).abs() < WIN)[None, None]; cond = None
        if C["adaln"]:
            with torch.autocast("cuda", enabled=False): cond = F.silu(s.temb @ s.wt1 + s.bt1) @ s.wt2 + s.bt2
        x = s.emb(ids); cos, sin = s.cos[:, :S], s.sin[:, :S]
        for b in s.blocks: x = b(x, cos, sin, km, lm, cond)
        h = s.nf(x)
        if cond is not None:
            with torch.autocast("cuda", enabled=False): sh, sc = (F.silu(cond) @ s.wadaf).chunk(2, -1)
            h = h * (1 + sc[:, None]) + sh[:, None]
        return (h * tm[..., None]).sum(1) / ln.clamp(min=1)[:, None]

class CJ(nn.Module):
    def __init__(s, enc):
        super().__init__(); s.enc = enc; s.att = nn.Linear(D, 1); s.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(D, NL))
    def forward(s, ids, ln, cm):
        B, Kc, T = ids.shape; f = s.enc(ids.view(B * Kc, T), ln.reshape(-1)).float().view(B, Kc, -1)
        a = s.att(f).squeeze(-1).masked_fill(~cm, -1e4).softmax(1)
        return s.head((a[..., None] * f).sum(1))

class LoRMA(nn.Module):
    def __init__(s, b, r=16): super().__init__(); s.b = b; o = b.out_features; s.A = nn.Parameter(torch.randn(o, r) * o ** -0.5); s.B = nn.Parameter(torch.zeros(r, o))
    def forward(s, x): y = s.b(x); return y + (y @ s.A) @ s.B

class Adapter(nn.Module):
    def __init__(s, b, r=64):
        super().__init__(); s.b = b; o = b.out_features; s.d, s.u = nn.Linear(o, r), nn.Linear(r, o); nn.init.zeros_(s.u.weight); nn.init.zeros_(s.u.bias)
    def forward(s, x): y = s.b(x); return y + s.u(F.gelu(s.d(y)))

class AdaLoRA(nn.Module):
    def __init__(s, b, r=12):
        super().__init__(); s.b = b; i, o = b.in_features, b.out_features
        s.P, s.Q, s.E = nn.Parameter(torch.empty(o, r)), nn.Parameter(torch.empty(r, i)), nn.Parameter(torch.zeros(r))
        nn.init.orthogonal_(s.P); nn.init.orthogonal_(s.Q); s.register_buffer("mk", torch.ones(r)); s.register_buffer("imp", torch.zeros(r))
    def forward(s, x): return s.b(x) + ((x @ s.Q.t()) * (s.E * s.mk)) @ s.P.t()

def orth(a):
    I = torch.eye(a.E.numel(), device=a.E.device); return ((a.P.t() @ a.P - I) ** 2).sum() + ((a.Q @ a.Q.t() - I) ** 2).sum()

def ada_step(ada, step, total, b0):
    for a in ada: a.imp.mul_(0.85).add_(0.15 * (a.E * a.E.grad).abs().detach())
    ti, tf = int(0.1 * total), int(0.7 * total)
    if ti <= step <= tf and (step % 25 == 0 or step == tf):
        bud = int(b0 // 2 + (b0 - b0 // 2) * (1 - (step - ti) / max(tf - ti, 1)) ** 3)
        thr = torch.cat([a.imp for a in ada]).topk(bud).values[-1]
        for a in ada: a.mk.copy_((a.imp >= thr).float())

def adapt(m, kind):
    if kind in ("full", "kd"): return m
    for p in m.parameters(): p.requires_grad = False
    for b in m.enc.blocks:
        for n in LINS:
            l = getattr(b, n)
            if kind == "bitfit": l.bias = nn.Parameter(torch.zeros(l.out_features))
            elif kind == "lorma": setattr(b, n, LoRMA(l))
            elif kind == "adalora": setattr(b, n, AdaLoRA(l))
        if kind == "adapter":
            for n in ("wo", "w2"): setattr(b, n, Adapter(getattr(b, n)))
    for p in [*m.att.parameters(), *m.head.parameters()]: p.requires_grad = True
    return m

def build(kind):
    m = adapt(CJ(Enc(range(0, L, 2) if kind == "kd" else range(L))), kind); m.head[1].bias.data = torch.from_numpy(np.log(PRI / (1 - PRI)).astype(np.float32)); return m

ours_mdlm_adaln_hybrid L=12 D=768 H=12 KV=4 HD=64 rows=16385 adaln=True hybrid=True | Tesla T4


## 4. Training utilities (DataParallel, fp16 autocast, class-weighted loss, time-budgeted cosine schedule)

In [4]:
@torch.no_grad()
def predict(dm, toks, bs=8):
    tr = dm.training; dm.eval(); out = []
    for i in range(0, len(toks), bs):
        ids, ln, cm = mk(toks, range(i, min(i + bs, len(toks))), False)
        with torch.autocast("cuda", torch.float16): out.append(torch.sigmoid(dm(ids, ln, cm).float()).cpu())
    dm.train(tr); return torch.cat(out).numpy()

TS = np.arange(0.02, 0.98, 0.02)
def lf1(q, y): return 2 * (q & y).sum(0) / np.maximum(q.sum(0) + y.sum(0), 1)
def msc(q, y):
    f = lf1(q, y); return dict(macro=float(f[y.sum(0) > 0].mean()), micro=float(2 * (q & y).sum() / max(q.sum() + y.sum(), 1)), subset=float((q == y).all(1).mean()), perl=f)
def tg(p, y):
    s = y.sum(0) > 0; return float(TS[int(np.argmax([lf1(p > t, y)[s].mean() for t in TS]))])
def tl(p, y, m=5.0):
    g = tg(p, y); b = TS[np.stack([lf1(p > t, y) for t in TS]).argmax(0)]; n = y.sum(0); w = n / (n + m)
    return np.where(n > 0, w * b + (1 - w) * g, g)
def dec(p, t):
    q = p > t; e = np.where(~q.any(1))[0]; q[e, (p[e] / t).argmax(1)] = True; return q
def xfit(p, y, fn):
    r = np.random.RandomState(0).permutation(len(p)); h = len(p) // 2; q = np.zeros_like(y)
    for a, b in ((r[:h], r[h:]), (r[h:], r[:h])): q[b] = dec(p[b], fn(p[a], y[a]))
    return msc(q, y)["macro"]
def pick(p, y):
    fns = dict(globalthr=tg, perlabel=tl); est = {k: xfit(p, y, f) for k, f in fns.items()}; k = max(est, key=est.get); return k, fns[k](p, y), est

def fit(name, m, kind, budget, teach=None):
    m.cuda().train(); dm = nn.DataParallel(m) if NG > 1 else m
    tp = [p for p in m.parameters() if p.requires_grad]; hp = [*m.att.parameters(), *m.head.parameters()]; hid = {id(p) for p in hp}
    opt = torch.optim.AdamW([dict(params=[p for p in tp if id(p) not in hid], base=LR[kind]), dict(params=hp, base=1e-3)], weight_decay=0.01, betas=(0.9, 0.98))
    sc = torch.amp.GradScaler("cuda"); pw = torch.from_numpy(PW).cuda()
    ada = [x for x in m.modules() if isinstance(x, AdaLoRA)]; b0 = sum(a.E.numel() for a in ada)
    step, total, tb, ptr, perm, ema, hist, t0 = 0, 10 ** 9, None, 0, np.random.permutation(NTR), None, [], time.time()
    torch.cuda.reset_peak_memory_stats()
    while step < total and time.time() - T0 < TOTAL_SEC - POST_SEC:
        if ptr + BS > len(perm): perm, ptr = np.random.permutation(NTR), 0
        idx = perm[ptr:ptr + BS]; ptr += BS; ids, ln, cm = mk(TR, idx, True); y = torch.from_numpy(Y[idx]).cuda()
        f = min(1.0, (step + 1) / 100) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(step / total, 1.0))))
        for g in opt.param_groups: g["lr"] = g["base"] * f
        with torch.autocast("cuda", torch.float16):
            lg = dm(ids, ln, cm).float(); loss = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)
            if teach is not None:
                with torch.no_grad(): tl_ = teach(ids, ln, cm).float()
                loss = 0.5 * loss + 2.0 * F.binary_cross_entropy_with_logits(lg / 2, torch.sigmoid(tl_ / 2))
        if ada: loss = loss + 0.1 * sum(orth(a) for a in ada)
        opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.unscale_(opt); gn = nn.utils.clip_grad_norm_(tp, 1.0)
        if ada and torch.isfinite(gn): ada_step(ada, step, total, b0)
        sc.step(opt); sc.update(); step += 1
        if total == 10 ** 9:
            if step == 5: torch.cuda.synchronize(); tb = time.time()
            if step == 25:
                torch.cuda.synchronize(); rate = (time.time() - tb) / 20; total = step + int(0.97 * (budget - (time.time() - t0)) / rate)
                print(f"[{name}] {rate:.3f}s/step -> total_steps={total} (~{total * BS / NTR:.2f} epochs)", flush=True)
        if step % 10 == 0: ema = loss.item() if ema is None else 0.9 * ema + 0.1 * loss.item()
        if step % 50 == 0 and ema is not None: hist.append((step, ema))
        if step % 200 == 0: print(f"[{name}] step {step}/{total} loss {ema:.4f} | {(time.time() - t0) / 60:.1f} min", flush=True)
        if total < 10 ** 9 and step < total and step % max(total // 5, 1) == 0:
            ps, ys = predict(dm, [DV[i] for i in SUB]), DYB[SUB]; print(f"[{name}] step {step} dev-subset macro-F1 {msc(dec(ps, tg(ps, ys)), ys)['macro']:.4f}", flush=True)
    tt = time.time() - t0; mem = torch.cuda.max_memory_allocated() / 2 ** 30
    t1 = time.time(); pv = predict(dm, DV); ds = time.time() - t1; pt = predict(dm, TE)
    mth, thr, est = pick(pv, DYB); r = msc(dec(pv, thr), DYB); qt = dec(pt, thr); sup = DYB.sum(0) > 0; ra = sup & (CNT < 10); fr = sup & (CNT >= 10)
    os.makedirs(f"{WORK}/{name}", exist_ok=True); pj = f"{WORK}/{name}/prediction.json"
    json.dump([{"id": i, "pred": [int(LAB[j]) for j in np.where(q)[0]]} for i, q in zip(TE_ID, qt)], open(pj, "w"))
    with zipfile.ZipFile(f"{WORK}/sub_{name}.zip", "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, "prediction.json")
    np.save(f"{WORK}/{name}/dev_probs.npy", pv); np.save(f"{WORK}/{name}/test_probs.npy", pt)
    torch.save({n: p.detach().cpu() for n, p in m.named_parameters() if p.requires_grad}, f"{WORK}/{name}/weights.pt")
    res = dict(name=name, f1=r["macro"], micro=r["micro"], subset=r["subset"], thr_method=mth, xfit=est, rare_f1=float(r["perl"][ra].mean()) if ra.any() else None, freq_f1=float(r["perl"][fr].mean()) if fr.any() else None,
               avg_labels_test=float(qt.sum(1).mean()), avg_labels_dev=float(DYB.sum(1).mean()), params_M=sum(p.numel() for p in m.parameters()) / 1e6, train_M=sum(p.numel() for p in tp) / 1e6,
               steps=step, train_min=tt / 60, mem_GB=mem, dev_infer_s=ds, perl=r["perl"].tolist(), hist=hist)
    print(f"[{name}] dev macro-F1 {r['macro']:.4f} | micro {r['micro']:.4f} | subset {r['subset']:.4f} | thr={mth} xfit={ {k: round(v, 4) for k, v in est.items()} } | rare {res['rare_f1']} freq {res['freq_f1']}", flush=True); return res, m

## 5. Experiments (time-shared budget; KD teacher = best dev macro-F1 model so far, student = every-other-layer Phase 1 init)

In [5]:
RES, BEST = {}, (-1.0, None)
for i, (name, w) in enumerate(EXPS):
    alloc = (TOTAL_SEC - (time.time() - T0)) * w / sum(x[1] for x in EXPS[i:])
    if alloc < 20 * 60: print("skip", name, "(not enough time)"); continue
    print(f"=== {name}: {alloc / 60:.0f} min allocated, {(time.time() - T0) / 60:.0f} min elapsed", flush=True)
    m, teach, tm = build(name), None, None
    if name == "kd": assert BEST[1] is not None; tm = BEST[1].cuda().eval(); teach = nn.DataParallel(tm) if NG > 1 else tm
    RES[name], m = fit(name, m, name, alloc - POST_SEC, teach)
    if name != "kd" and RES[name]["f1"] > BEST[0]: BEST = (RES[name]["f1"], copy.deepcopy(m).cpu())
    if tm is not None: tm.cpu()
    m.cpu(); del m, teach, tm; gc.collect(); torch.cuda.empty_cache()
    json.dump(RES, open(f"{WORK}/results.json", "w"), indent=1)

print(f"{'exp':9s}{'total M':>9s}{'train M':>9s}{'macro':>8s}{'micro':>8s}{'subset':>8s}{'rare':>8s}{'freq':>8s}{'thr':>11s}{'min':>7s}{'GB':>6s}{'dev s':>7s}")
for k, r in RES.items(): print(f"{k:9s}{r['params_M']:9.1f}{r['train_M']:9.2f}{r['f1']:8.4f}{r['micro']:8.4f}{r['subset']:8.4f}{(r['rare_f1'] or 0):8.4f}{(r['freq_f1'] or 0):8.4f}{r['thr_method']:>11s}{r['train_min']:7.1f}{r['mem_GB']:6.1f}{r['dev_infer_s']:7.0f}")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
plt.figure(figsize=(9, 5))
for k, r in RES.items():
    if r["hist"]: plt.plot(*zip(*r["hist"]), label=k)
plt.xlabel("step"); plt.ylabel("train loss (EMA)"); plt.grid(True); plt.legend(); plt.savefig(f"{WORK}/loss_curves.png", dpi=200, bbox_inches="tight"); plt.show()
print("submissions:", sorted(glob.glob(f"{WORK}/sub_*.zip")))

=== lorma: 64 min allocated, 1 min elapsed
[lorma] 0.637s/step -> total_steps=5147 (~0.48 epochs)
[lorma] step 200/5147 loss 0.4321 | 2.3 min
[lorma] step 400/5147 loss 0.3974 | 4.5 min
[lorma] step 600/5147 loss 0.3963 | 6.8 min
[lorma] step 800/5147 loss 0.3538 | 9.0 min
[lorma] step 1000/5147 loss 0.3905 | 11.3 min
[lorma] step 1029 dev-subset macro-F1 0.2825
[lorma] step 1200/5147 loss 0.3406 | 14.0 min
[lorma] step 1400/5147 loss 0.2905 | 16.2 min
[lorma] step 1600/5147 loss 0.3032 | 18.5 min
[lorma] step 1800/5147 loss 0.3173 | 20.7 min
[lorma] step 2000/5147 loss 0.3260 | 23.0 min
[lorma] step 2058 dev-subset macro-F1 0.2873
[lorma] step 2200/5147 loss 0.2905 | 25.7 min
[lorma] step 2400/5147 loss 0.2991 | 27.9 min
[lorma] step 2600/5147 loss 0.3152 | 30.2 min
[lorma] step 2800/5147 loss 0.3095 | 32.5 min
[lorma] step 3000/5147 loss 0.3069 | 34.7 min
[lorma] step 3087 dev-subset macro-F1 0.3232
[lorma] step 3200/5147 loss 0.2888 | 37.4 min
[lorma] step 3400/5147 loss 0.2830 | 39

## 6. Long-document analysis: which part of the judgment carries the signal (best non-KD model, dev set)

In [6]:
bm = BEST[1].cuda().eval(); dm = nn.DataParallel(bm) if NG > 1 else bm
for SELMODE in ("even", "head", "tail"):
    pv = predict(dm, DV); k, thr, est = pick(pv, DYB); r = msc(dec(pv, thr), DYB); print(f"chunks={SELMODE:5s} macro {r['macro']:.4f} micro {r['micro']:.4f} subset {r['subset']:.4f} | thr={k} xfit { {a: round(b, 4) for a, b in est.items()} }")
SELMODE = "even"; pv = predict(dm, DV); k, thr, est = pick(pv, DYB); q = dec(pv, thr); n = DYB.sum(0); miss = (DYB & ~q).sum(0) / np.maximum(n, 1); fp = (q & ~DYB).sum(0)
for nm, t in (("global", tg(pv, DYB)), ("per-label", tl(pv, DYB))): print(f"{nm:9s} thr: macro {msc(dec(pv, t), DYB)['macro']:.4f} (dev-tuned, optimistic) | avg labels/doc {dec(pv, t).sum(1).mean():.2f} vs gold {DYB.sum(1).mean():.2f}")
print("most missed (dev support>=5): id, support, miss-rate", [(LAB[j], int(n[j]), round(float(miss[j]), 2)) for j in np.argsort(-miss * (n >= 5))[:10]])
print("most false positives: id, count", [(LAB[j], int(fp[j])) for j in np.argsort(-fp)[:10]])
bm.cpu()

chunks=even  macro 0.3788 micro 0.4730 subset 0.0304 | thr=perlabel xfit {'globalthr': 0.3545, 'perlabel': 0.3656}
chunks=head  macro 0.3679 micro 0.4588 subset 0.0250 | thr=perlabel xfit {'globalthr': 0.3453, 'perlabel': 0.3542}
chunks=tail  macro 0.3709 micro 0.4626 subset 0.0274 | thr=perlabel xfit {'globalthr': 0.3477, 'perlabel': 0.3572}
global    thr: macro 0.3552 (dev-tuned, optimistic) | avg labels/doc 4.64 vs gold 3.92
per-label thr: macro 0.3788 (dev-tuned, optimistic) | avg labels/doc 5.18 vs gold 3.92
most missed (dev support>=5): id, support, miss-rate [(2, 98, 0.91), (0, 86, 0.91), (27, 83, 0.9), (8, 92, 0.9), (18, 100, 0.84), (40, 152, 0.81), (52, 272, 0.81), (84, 68, 0.79), (57, 115, 0.79), (65, 105, 0.79)]
most false positives: id, count [(97, 2014), (5, 1807), (43, 1604), (39, 1290), (51, 1064), (13, 804), (10, 760), (44, 697), (96, 690), (46, 653)]


CJ(
  (enc): Enc(
    (emb): Embedding(16385, 768)
    (blocks): ModuleList(
      (0-11): 12 x Blk(
        (wq): LoRMA(
          (b): Linear(in_features=768, out_features=768, bias=False)
        )
        (wk): LoRMA(
          (b): Linear(in_features=768, out_features=256, bias=False)
        )
        (wv): LoRMA(
          (b): Linear(in_features=768, out_features=256, bias=False)
        )
        (wo): LoRMA(
          (b): Linear(in_features=768, out_features=768, bias=False)
        )
        (w1): LoRMA(
          (b): Linear(in_features=768, out_features=2048, bias=False)
        )
        (w3): LoRMA(
          (b): Linear(in_features=768, out_features=2048, bias=False)
        )
        (w2): LoRMA(
          (b): Linear(in_features=2048, out_features=768, bias=False)
        )
        (n1): RMS()
        (n2): RMS()
        (qn): RMS()
        (kn): RMS()
      )
    )
    (nf): RMS()
  )
  (att): Linear(in_features=768, out_features=1, bias=True)
  (head): Sequential(
